In [1]:
import os
import time
import warnings

import torch
import pickle
import numpy as np
import xarray as xr
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from utils import read_xarray
from model import SimpleTanhNN
from sklearn_som.som import SOM

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Check for GPU availability and set the device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Data Preparation

In [3]:
# Define constants and hyperparameters
data_type = "CESM"
data_num = "001"
batch_size = 32
num_hiddens = 100
normalize = True
y_normalize = False
base_learning_rate = 1e-2
lr = base_learning_rate * batch_size / 256
weight_decay = 0
test_method = "future"
test_ratio = 0.2

In [4]:
rdir = os.getcwd()
dir_name = rdir + "/data"

In [5]:
# Load data using the custom read_xarray function
chl, mld, sss, sst, u10, xco2, icefrac, patm, pco2 = read_xarray(dir_name, data_type, data_num)

# Apply masks and preprocessing to the data
socat_mask = pco2.pCO2_socat.data[:420, ::-1] > 0.5

sst = sst.SST.data[:420, ::-1]
chl = chl.Chl.data[:420, ::-1]
mld = mld.MLD.data[:420, ::-1]
sss = sss.SSS.data[:420, ::-1]
xco2 = xco2.XCO2.data[:420, np.newaxis, np.newaxis]
xco2 = np.broadcast_to(xco2, (420, 180, 360))
pco2 = pco2.pCO2.data[:420, ::-1]

land_mask = xr.open_dataset(dir_name + "/land.nc").mask.data[::-1]
land_mask = (~np.isnan(land_mask))[np.newaxis, :, :]
land_mask = np.broadcast_to(land_mask, (420, 180, 360))
land_mask = land_mask & (pco2 > 0.5) & (chl > 0)

socat_mask = land_mask & socat_mask

In [6]:
# Function to remove seasonal patterns from a 3D data array
def deseasonalize_3d_array(data):
    n_months = 12
    n_years = data.shape[0] // n_months
    latitudes = data.shape[1]
    longitudes = data.shape[2]

    reshaped_data = data.reshape(n_years, n_months, latitudes, longitudes)
    monthly_means = reshaped_data.mean(axis=0)
    deseasonalized_data = (reshaped_data - monthly_means).reshape(data.shape[0], latitudes, longitudes)

    return deseasonalized_data

In [7]:
# Deseasonalize data
sst_ds = deseasonalize_3d_array(sst)
chl_ds = deseasonalize_3d_array(chl)
mld_ds = deseasonalize_3d_array(mld)
sss_ds = deseasonalize_3d_array(sss)
xco2_ds = deseasonalize_3d_array(xco2)
logchl = np.log(chl)
logmld = np.log(mld)

In [8]:
X = np.stack((sst, logchl, logmld, sss, xco2, sst_ds, chl_ds, mld_ds, sss_ds, xco2_ds), axis=-1)

if normalize:
    X_mean = np.mean(X[land_mask], axis=0)
    X_std = np.std(X[land_mask], axis=0)
    X = (X - X_mean) / X_std

In [9]:
# Split the data into training and testing sets
if test_method == 'future':
    l = len(X)
    X_train = X[:int(l * (1 - test_ratio))]
    X_test = X[int(l * (1 - test_ratio)):]
    socat_mask_train = socat_mask[:int(l * (1 - test_ratio))]
    socat_mask_test = socat_mask[int(l * (1 - test_ratio)):]
    y_train = pco2[:int(l * (1 - test_ratio))]
    y_test = pco2[int(l * (1 - test_ratio)):]
    land_mask_train = land_mask[:int(l * (1 - test_ratio))]
    land_mask_test = land_mask[int(l * (1 - test_ratio)):]
else:
    raise NotImplementedError

# Train SOM

In [10]:
X_pixel_train = X_train[land_mask_train]
X_pixel_test = X_test[land_mask_test]
y_pixel_train = y_train[land_mask_train]
y_pixel_test = y_test[land_mask_test]

In [11]:
# Train the Self-Organizing Map (SOM) model
SOM_model = SOM(m=4, n=4, dim=4)
SOM_input = np.stack((sst, logmld, sss, pco2), axis=-1)[:int(l * (1 - test_ratio))][land_mask_train]
SOM_model.fit(SOM_input)

In [12]:
with open('SOM_model.pkl', 'wb') as f:
    pickle.dump(SOM_model, f)

# Prepare Dataset for FFN

In [13]:
SOM_socat_train = np.stack((sst, logmld, sss, pco2), axis=-1)
SOM_socat_test = SOM_socat_train[int(l * (1 - test_ratio)):][socat_mask_test]
SOM_socat_train = SOM_socat_train[:int(l * (1 - test_ratio))][socat_mask_train]

X_socat_train = X_train[socat_mask_train]
X_socat_test = X_test[socat_mask_test]
y_socat_train = y_train[socat_mask_train]
y_socat_test = y_test[socat_mask_test]

In [14]:
province_train = SOM_model.predict(SOM_socat_train)
province_test = SOM_model.predict(SOM_socat_test)

In [15]:
# Prepare datasets and dataloaders for each province
train_datasets = []
test_datasets = []
train_iters = []
test_iters = []
y_means = []
y_stds = []

for province_id in range(16):
    train_mask = (province_train == province_id)
    test_mask = (province_test == province_id)

    temp_y = y_socat_train[train_mask]
    y_means.append(np.mean(temp_y))
    y_stds.append(np.std(temp_y))
    train_datasets.append(TensorDataset(
        torch.tensor(X_socat_train[train_mask], dtype=torch.float64),
        torch.tensor(temp_y, dtype=torch.float64)
    ))
    test_datasets.append(TensorDataset(
        torch.tensor(X_socat_test[test_mask], dtype=torch.float64),
        torch.tensor(y_socat_test[test_mask], dtype=torch.float64)
    ))

    train_iters.append(DataLoader(train_datasets[-1], batch_size=batch_size, shuffle=True))
    test_iters.append(DataLoader(test_datasets[-1], batch_size=batch_size, shuffle=False))

In [16]:
# Initialize neural networks and optimizers for each province
FFNs = []
optims = []
for i in range(16):
    FFNs.append(SimpleTanhNN(10, num_hiddens, 1).double().to(device))
    optims.append(torch.optim.SGD(FFNs[-1].parameters(), lr=lr, weight_decay=weight_decay))

criterion = nn.MSELoss(reduction='mean')
train_losses = []
test_losses = []

# Train FFN

In [17]:
# Function to train the neural network
def train(net, train_iter, test_iter, num_epochs, optimizer, criterion, y_normalize, y_mean, y_std, device,
          ltrain, ltest):
    # Function to train for one epoch
    def train_one_epoch(net, device, data_iter, optimizer, criterion, y_normalize, y_mean, y_std):
        train_l_sum, train_l2_sum = torch.zeros(1, device=device), torch.zeros(1, device=device)

        for X, y in data_iter:
            l = X.size(0)
            X_cut = X.to(device)
            X = X_cut
            ypred = net(X)
            if y_normalize:
                y_norm = (y - y_mean) / y_std
                loss = criterion(ypred, torch.unsqueeze(y_norm, 1).to(device))
                ypred = ypred * y_std + y_mean
                loss2 = criterion(ypred, torch.unsqueeze(y, 1).to(device))
                train_l2_sum += loss2.detach() * l
            else:
                loss = criterion(ypred, torch.unsqueeze(y, 1).to(device))

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            train_l_sum += loss.detach() * l

        return train_l_sum.cpu().item(), train_l2_sum.cpu().item()

    # Function to evaluate the model accuracy
    def evaluate_accuracy(net, device, data_iter, criterion, y_normalize, y_mean, y_std):
        loss_sum, loss2_sum = torch.zeros(1, device=device), torch.zeros(1, device=device)
        with torch.no_grad():
            net.eval()
            for X, y in data_iter:
                l = X.size(0)
                X_cut = X.to(device)
                X = X_cut
                ypred = net(X)
                if y_normalize:
                    y_norm = (y - y_mean) / y_std
                    loss = criterion(ypred, torch.unsqueeze(y_norm, 1).to(device))
                    ypred = ypred * y_std + y_mean
                    loss2 = criterion(ypred, torch.unsqueeze(y, 1).to(device))
                    loss2_sum += loss2.detach() * l
                else:
                    loss = criterion(ypred, torch.unsqueeze(y, 1).to(device))

                loss_sum += loss.detach() * l
            net.train()
        return loss_sum.cpu().item(), loss2_sum.cpu().item()

    optimizer.zero_grad()

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, train_loss2 = train_one_epoch(net, device, train_iter, optimizer, criterion, y_normalize,
                                                  y_mean, y_std)
        train_loss /= ltrain
        train_loss2 /= ltrain

        if test_iter is not None:
            test_loss, test_loss2 = evaluate_accuracy(net, device, test_iter, criterion, y_normalize, y_mean, y_std)
            test_loss /= ltest
            test_loss2 /= ltest

        print("epoch %d: train loss %.4f, train loss2 %.4f, test loss %.4f, test loss2 %.4f time %.1f sec" % (
        epoch + 1, train_loss, train_loss2, test_loss, test_loss2, time.time() - start))

    return train_loss, train_loss2, test_loss, test_loss2

In [18]:
# Train the neural networks
for i in range(0, 15):
    ltrain = np.sum(province_train == i)
    ltest = np.sum(province_test == i)
    train_loss, train_loss2, test_loss, test_loss2 = train(FFNs[i], train_iters[i], test_iters[i], 10,
                                                           optims[i], criterion, y_normalize,
                                                           y_means[i], y_stds[i], device, ltrain, ltest)
    train_losses.append(train_loss)
    test_losses.append(test_loss)

i = 15
ltrain = np.sum(province_train == i)
ltest = np.sum(province_test == i)
train_loss, train_loss2, test_loss, test_loss2 = train(FFNs[i], train_iters[i], test_iters[i], 50,
                                                       optims[i], criterion, y_normalize,
                                                       y_means[i], y_stds[i], device, ltrain, ltest)
train_losses.append(train_loss)
test_losses.append(test_loss)

epoch 1: train loss 1929.8353, train loss2 0.0000, test loss 345.8477, test loss2 0.0000 time 1.7 sec
epoch 2: train loss 469.6815, train loss2 0.0000, test loss 340.0127, test loss2 0.0000 time 1.6 sec
epoch 3: train loss 407.2140, train loss2 0.0000, test loss 363.6517, test loss2 0.0000 time 1.6 sec
epoch 4: train loss 364.7043, train loss2 0.0000, test loss 360.1018, test loss2 0.0000 time 1.7 sec
epoch 5: train loss 329.2110, train loss2 0.0000, test loss 322.6099, test loss2 0.0000 time 1.8 sec
epoch 6: train loss 308.6824, train loss2 0.0000, test loss 331.2706, test loss2 0.0000 time 1.8 sec
epoch 7: train loss 295.5149, train loss2 0.0000, test loss 347.5021, test loss2 0.0000 time 1.9 sec
epoch 8: train loss 284.2568, train loss2 0.0000, test loss 331.8818, test loss2 0.0000 time 1.8 sec
epoch 9: train loss 270.8321, train loss2 0.0000, test loss 318.8319, test loss2 0.0000 time 1.8 sec
epoch 10: train loss 266.8582, train loss2 0.0000, test loss 313.4089, test loss2 0.0000 t

# Prepare Dataset for Calibration

In [19]:
all_features = np.stack((sst, logmld, sss, pco2), axis=-1)
reshaped_features = all_features.reshape(-1, 4)
all_provinces_flat = SOM_model.predict(reshaped_features)
all_provinces = all_provinces_flat.reshape((420, 180, 360))
all_provinces[~land_mask] = -1

In [20]:
# Save the province predictions
with open('all_provinces.pkl', 'wb') as f:
    pickle.dump(all_provinces, f)

In [21]:
raw_y_hat = np.zeros((420, 180, 360))
with torch.no_grad():
    for province in range(len(FFNs)):
        mask = (all_provinces == province)
        indices = np.argwhere(mask)

        if indices.size > 0:
            batch_indices = indices[:, 0], indices[:, 1], indices[:, 2]
            batch_inputs = torch.tensor(X[batch_indices], dtype=torch.float64, device=device)
            batch_outputs = FFNs[province](batch_inputs).cpu().numpy()
            raw_y_hat[mask] = batch_outputs.flatten()

In [22]:
# Save the predictions
with open('raw_y_hat.pkl', 'wb') as f:
    pickle.dump(raw_y_hat, f)

# Prepare CESM 002 Dataset

In [23]:
# Load additional data for further predictions
chl_CESM002, mld_CESM002, sss_CESM002, sst_CESM002, u10_CESM002, xco2_CESM002, icefrac_CESM002,\
patm_CESM002, pco2_CESM002 = read_xarray(dir_name, 'CESM', '002')
pco2_CESM002 = pco2_CESM002.pCO2.data[:420, ::-1]
sst_CESM002 = sst_CESM002.SST.data[:420, ::-1]
mld_CESM002 = mld_CESM002.MLD.data[:420, ::-1]
sss_CESM002 = sss_CESM002.SSS.data[:420, ::-1]
logmld_CESM002 = np.log(mld_CESM002)

In [24]:
all_features_CESM002 = np.stack((sst_CESM002, logmld_CESM002, sss_CESM002, pco2_CESM002), axis=-1)
reshaped_features_CESM002 = all_features_CESM002.reshape(-1, 4)
all_provinces_flat_CESM002 = SOM_model.predict(reshaped_features_CESM002)
all_provinces_CESM002 = all_provinces_flat_CESM002.reshape((420, 180, 360))
all_provinces_CESM002[~land_mask] = -1

In [25]:
# Save the province predictions for the additional data
with open('all_provinces_CESM002.pkl', 'wb') as f:
    pickle.dump(all_provinces_CESM002, f)